In [17]:
import cobra

In [18]:
model = cobra.io.read_sbml_model("model.xml")

In [19]:
new_rxn = cobra.Reaction("ec7211_c0")
new_rxn.name = "NADH:ubiquinone reductase (Na+-transporting)"
new_rxn.add_metabolites({
    model.metabolites.cpd00004_c0: -1,  # NADH
    model.metabolites.cpd00067_c0: -1,  # H+
    model.metabolites.cpd15560_c0: -1,  # Ubiquinone
    model.metabolites.cpd00971_c0: -2,  # Sodium ion
    model.metabolites.cpd00003_c0: 1,   # NAD+
    model.metabolites.cpd15561_c0: 1,   # Ubiquinol
    model.metabolites.cpd00971_e0: 2,   # Sodium ion (extracellular)
})
new_rxn.compartment = "c0"
new_rxn.lower_bound = 0
new_rxn.upper_bound = 1000
# Add GPR
# Assign the gene reaction rule string, will automatically create the corresponding gene objects
new_rxn.gene_reaction_rule = "( WP_049586994.1 and WP_014950797.1 and WP_049586957.1 and WP_014950795.1 and WP_014950794.1 and WP_014950793.1 )"
# Add EC number
new_rxn.annotation["ec-code"] = "7.2.1.1"


In [20]:
new_rxn

Reaction identifier,ec7211_c0
Name,NADH:ubiquinone reductase (Na+-transporting)
Memory address,0x11a8b9e50
Stoichiometry,cpd00004_c0 + cpd00067_c0 + 2 cpd00971_c0 + cpd15560_c0 --> cpd00003_c0 + 2 cpd00971_e0 + cpd15561_c0 NADH + H+ + 2 Na+ + Ubiquinone-8 --> NAD + 2 Na+ [e0] + Ubiquinol-8
GPR,WP_049586994.1 and WP_014950797.1 and WP_049586957.1 and WP_014950795.1 and WP_014950794.1 and...
Lower bound,0
Upper bound,1000


In [29]:
model.add_reactions([new_rxn])

In [31]:
# Change the names of the gene products
model.genes.get_by_id('WP_049586994.1').name = "nqrA"
model.genes.get_by_id('WP_014950797.1').name = "nqrB"
model.genes.get_by_id('WP_049586957.1').name = "nqrC"
model.genes.get_by_id('WP_014950795.1').name = "nqrD"
model.genes.get_by_id('WP_014950794.1').name = "nqrE"
model.genes.get_by_id('WP_014950793.1').name = "nqrF"

In [32]:
cobra.io.write_sbml_model(model, "model.xml")

In [33]:
cobra.io.save_json_model(model, "/Users/helenscott/Desktop/model.json")

# Debug Energy Generating Cycles

The MEMOTE test uses FBA, but I want pFBA...

In [61]:
# Make a media without any carbon source
minimal_media = {
    "EX_cpd00067_e0": 1000,  # H+_e0
    "EX_cpd00058_e0": 1000,  # Cu2+_e0
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00971_e0": 1000,  # Na+_e0
    "EX_cpd00063_e0": 1000,  # Ca2+_e0
    "EX_cpd00048_e0": 1000,  # Sulfate_e0
    "EX_cpd10516_e0": 1000,  # fe3_e0
    "EX_cpd00254_e0": 1000,  # Mg_e0
    "EX_cpd00009_e0": 1000,  # Phosphate_e0
    "EX_cpd00205_e0": 1000,  # K+_e0
    "EX_cpd00013_e0": 1000,  # NH3_e0
    "EX_cpd00099_e0": 1000,  # Cl-_e0
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
    "EX_cpd00075_e0": 1000,  # Nitrite_e0
    "EX_cpd00001_e0": 1000,  # H2O_e0
    "EX_cpd00034_e0": 1000,  # Zn2+_e0
    "EX_cpd00149_e0": 1000,  # Co2+_e0
}
# Set it as the model's medium
model.medium = minimal_media

In [62]:
# Set the ATP maintenance reaction as the objective
model.objective = "rxn00062_c0"

In [63]:
# Run pFBA
solution = cobra.flux_analysis.pfba(model)

In [65]:
# Filter solution.fluxes to only include reactions with non-zero flux
rxns_to_check = solution.fluxes[solution.fluxes > 1]

In [69]:
# Print a mrakdown table of the reactions with non-zero flux
print("| Reaction ID | Reaction Name | Reaction | Flux Value |")
print("|-------------|---------------|---------|------------|")
for rxn_id in rxns_to_check.index:
    rxn = model.reactions.get_by_id(rxn_id)
    flux_value = solution.fluxes[rxn_id]
    print(f"| {rxn_id} | {rxn.name} | {rxn.build_reaction_string(use_metabolite_names=True)} | {flux_value:.2f} |")

| Reaction ID | Reaction Name | Reaction | Flux Value |
|-------------|---------------|---------|------------|
| rxn05209_c0 | TRANS-RXNBWI-115525.ce.maizeexp.NA+_NA+ [c0] | H+ + Na+ [e0] <=> H+ [e0] + Na+ | 1000.00 |
| rxn00083_c0 | NADPH:NAD+ oxidoreductase [c0] | NAD + NADPH --> NADH + NADP | 1000.00 |
| rxn00085_c0 | L-Glutamate:NADP+ oxidoreductase (transaminating) [c0] | NADP + 2.0 L-Glutamate <=> NADPH + 2-Oxoglutarate + L-Glutamine + H+ | 1000.00 |
| rxn08173_c0 | F(1)-ATPase [c0] | ADP + Phosphate + 4.0 H+ [e0] <=> H2O + ATP + 3.0 H+ | 500.00 |
| rxn14412_c0 | cytochrome-c reductase (ubiquinol: 2 protons) [c0] | 2.0 Cytochrome c3+ + Ubiquinol-8 --> 2.0 H+ [e0] + 2.0 Cytochrome c2+ + Ubiquinone-8 | 1000.00 |
| rxn08062_c0 | Na+/Acetate symport (periplasm) [c0] | Acetate [e0] + Na+ [e0] <=> Acetate + Na+ | 1000.00 |
| rxn00062_c0 | ATP hydrolysis | H2O + ATP --> ADP + Phosphate + H+ | 500.00 |
| rxn14427_c0 | Nitrate:cytochrome oxidoreductase [c0] | 2.0 H+ [e0] + 2.0 Cytochrome 

In [58]:
# Save the fluxes to a JSON file for Escher
solution.fluxes.to_json("/Users/helenscott/Desktop/fluxes.json")